In [23]:
import pandas as pd
import numpy as np
from collections import defaultdict

df = pd.read_csv("./data/processed/atp_matches.csv")

df["tourney_date"] = pd.to_datetime(
    df["tourney_date"]
)

winner_col = "winner_id" if "winner_id" in df.columns else "winner_name"
loser_col = "loser_id" if "loser_id" in df.columns else "loser_name"

df = df.dropna(subset=["tourney_date", winner_col, loser_col])

print("rows after cleaning:", len(df))
print(df[["tourney_date", winner_col, loser_col]].head())

rows after cleaning: 21223
  tourney_date  winner_id  loser_id
0   2006-01-16     103908    103632
1   2006-01-16     103990    103484
2   2006-01-16     103206    103909
3   2006-01-16     102148    103292
4   2006-01-16     103970    104339


In [ ]:
sort_cols = ["tourney_date"]
if "tourney_id" in df.columns:
    sort_cols.append("tourney_id")
if "match_num" in df.columns:
    sort_cols.append("match_num")

df = df.sort_values(sort_cols).reset_index(drop=True)

INITIAL_ELO = 1500.0
K = 32.0

ratings = defaultdict(lambda: INITIAL_ELO)
matches_played = defaultdict(int)

df["winner_elo_pre"] = np.nan
df["loser_elo_pre"] = np.nan
df["elo_prob_winner_pre"] = np.nan


def elo_expected(r_a, r_b):
    return 1 / (1 + 10 ** ((r_b - r_a) / 400))


for i, row in df.iterrows():
    winner = row[winner_col]
    loser = row[loser_col]

    winner_elo = ratings[winner]
    loser_elo = ratings[loser]

    p_winner = elo_expected(winner_elo, loser_elo)

    df.at[i, "winner_elo_pre"] = winner_elo
    df.at[i, "loser_elo_pre"] = loser_elo
    df.at[i, "elo_prob_winner_pre"] = p_winner

    ratings[winner] = winner_elo + K * (1 - p_winner)
    ratings[loser] = loser_elo + K * (0 - (1 - p_winner))

    matches_played[winner] += 1
    matches_played[loser] += 1

print("rows after Elo:", len(df))
print(df[["winner_elo_pre", "loser_elo_pre", "elo_prob_winner_pre"]].head())


rows after Elo: 21223
   winner_elo_pre  loser_elo_pre  elo_prob_winner_pre
0          1500.0         1500.0                  0.5
1          1500.0         1500.0                  0.5
2          1500.0         1500.0                  0.5
3          1500.0         1500.0                  0.5
4          1500.0         1500.0                  0.5


In [25]:
rng = np.random.default_rng(42)

model_df = df[df["tourney_date"] >= "2011-01-01"].copy()

swap = rng.random(len(model_df)) < 0.5

model_df["elo_a_pre"] = np.where(
    swap,
    model_df["winner_elo_pre"],
    model_df["loser_elo_pre"]
)

model_df["elo_b_pre"] = np.where(
    swap,
    model_df["loser_elo_pre"],
    model_df["winner_elo_pre"]
)

model_df["y"] = np.where(swap, 1, 0)
model_df["elo_diff"] = model_df["elo_a_pre"] - model_df["elo_b_pre"]

print("model rows:", len(model_df))
print(model_df[["elo_a_pre", "elo_b_pre", "elo_diff", "y"]].head())

model rows: 15423
        elo_a_pre    elo_b_pre    elo_diff  y
5800  1385.944553  2070.753505 -684.808951  0
5801  1391.131713  1471.311160  -80.179447  1
5802  1612.290810  1478.623444  133.667366  0
5803  1481.696402  1580.632701  -98.936299  0
5804  1595.574876  1526.077965   69.496911  1


In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, roc_auc_score

split = int(len(model_df) * 0.8)

train = model_df.iloc[:split]
test = model_df.iloc[split:]

X_train = train[["elo_diff"]]
y_train = train["y"]

X_test = test[["elo_diff"]]
y_test = test["y"]

print("train shape:", X_train.shape)
print("test shape:", X_test.shape)

logreg = LogisticRegression()
logreg.fit(X_train, y_train)

proba = logreg.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, pred))
print("AUC:", roc_auc_score(y_test, proba))
print("Log loss:", log_loss(y_test, proba))
print("Brier:", brier_score_loss(y_test, proba))

train shape: (12338, 1)
test shape: (3085, 1)
Accuracy: 0.6405186385737439
AUC: 0.7028272422092995
Log loss: 0.6249631854473972
Brier: 0.21824606778976197
